# Student Performance Analysis using Apache Spark and HDFS
**Dataset:** StudentsPerformance.csv (1000 students: gender, race/ethnicity, parental education, lunch, test prep, math/reading/writing scores)

**Pipeline:** Dataset → HDFS → Apache Spark (DataFrame + Spark SQL) → results saved back to HDFS → **interactive website (last cell)**

Run all cells in order. The final cell prints a public website link.

## 0. Setup

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Student Performance Analysis") \
    .getOrCreate()
print("Spark Version:", spark.version)

## HDFS Setup (single-node, pseudo-distributed)

In [ ]:
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz
!rm -rf /usr/local/hadoop && mv hadoop-3.3.6 /usr/local/hadoop
print("Hadoop installed")

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/usr/local/hadoop"
os.environ["HADOOP_CONF_DIR"] = "/usr/local/hadoop/etc/hadoop"
os.environ["PATH"] += ":/usr/local/hadoop/bin:/usr/local/hadoop/sbin"
os.environ["HDFS_NAMENODE_USER"] = "root"
os.environ["HDFS_DATANODE_USER"] = "root"
os.environ["HDFS_SECONDARYNAMENODE_USER"] = "root"

conf = "/usr/local/hadoop/etc/hadoop"
with open(f"{conf}/core-site.xml", "w") as f:
    f.write("""<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>""")
with open(f"{conf}/hdfs-site.xml", "w") as f:
    f.write("""<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file:///content/hdfs/namenode</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file:///content/hdfs/datanode</value></property>
  <property><name>dfs.permissions.enabled</name><value>false</value></property>
</configuration>""")
print("HDFS configured")

In [ ]:
!hdfs namenode -format -force > /dev/null 2>&1
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!sleep 15
!hdfs dfsadmin -safemode leave
!hdfs dfsadmin -report | head -20

## 1. Upload dataset and load it from HDFS into Spark

In [ ]:
import os, pandas as pd

path = "/content/StudentsPerformance.csv"
if not os.path.exists(path):
    from google.colab import files
    print("Please select StudentsPerformance.csv")
    up = files.upload()
    name = list(up.keys())[0]
    os.replace(name, path)

pdf = pd.read_csv(path)
# Spark-friendly column names (no spaces or slashes)
pdf.columns = ["gender", "race_ethnicity", "parental_education", "lunch",
               "test_prep", "math_score", "reading_score", "writing_score"]
pdf.to_csv("/content/students_clean.csv", index=False)
print("Rows:", len(pdf), "| Columns:", len(pdf.columns))

In [ ]:
!hdfs dfs -mkdir -p /user/student/input
!hdfs dfs -put -f /content/students_clean.csv /user/student/input/
!hdfs dfs -ls /user/student/input

df = spark.read.csv(
    "hdfs://localhost:9000/user/student/input/students_clean.csv",
    header=True,
    inferSchema=True
)
df.show(5)

## 2. Dataset Overview

In [ ]:
print("Number of Students:", df.count())
print("Number of Columns:", len(df.columns))
df.printSchema()

## 3. Statistical Summary

In [ ]:
df.describe().show()

## 4. Data Cleaning

In [ ]:
from pyspark.sql.functions import col, sum as _sum

df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

print("Records before removing duplicates:", df.count())
df = df.dropDuplicates()
print("Records after removing duplicates:", df.count())

## 5. Feature Engineering (average score, performance level, pass/fail)

In [ ]:
from pyspark.sql.functions import when, round as _round

df = df.withColumn("average_score",
        _round((col("math_score") + col("reading_score") + col("writing_score")) / 3, 2))

df = df.withColumn(
    "performance",
    when(col("average_score") >= 80, "Excellent")
    .when(col("average_score") >= 60, "Good")
    .when(col("average_score") >= 40, "Average")
    .otherwise("Poor")
)

# Passed = at least 40 in every subject
df = df.withColumn(
    "passed",
    (col("math_score") >= 40) & (col("reading_score") >= 40) & (col("writing_score") >= 40)
)
df.show(5)

## 6. Basic Performance Analysis

In [ ]:
from pyspark.sql.functions import avg, max as _max, min as _min, count

df.select(
    avg("math_score").alias("Avg Math"),
    avg("reading_score").alias("Avg Reading"),
    avg("writing_score").alias("Avg Writing")
).show()

df.select(
    _max("average_score").alias("Highest Average"),
    _min("average_score").alias("Lowest Average")
).show()

## 7. Top Performing Students

In [ ]:
df.orderBy(col("average_score").desc()).show(10)

## 8. Group-wise Analysis (gender, lunch, test prep, parental education, race)

In [ ]:
for c in ["gender", "lunch", "test_prep", "parental_education", "race_ethnicity"]:
    print("=== Average scores by", c, "===")
    df.groupBy(c).agg(
        avg("math_score").alias("math"),
        avg("reading_score").alias("reading"),
        avg("writing_score").alias("writing"),
        avg("average_score").alias("overall"),
        count("*").alias("students")
    ).orderBy("overall", ascending=False).show(truncate=False)

## 9. Performance Categories

In [ ]:
df.groupBy("performance").count().orderBy("count", ascending=False).show()
df.groupBy("passed").count().show()

## 10. Spark SQL Analysis

In [ ]:
df.createOrReplaceTempView("students")

In [ ]:
spark.sql("""
    SELECT test_prep, ROUND(AVG(average_score), 2) AS avg_score, COUNT(*) AS students
    FROM students
    GROUP BY test_prep
""").show()

In [ ]:
spark.sql("""
    SELECT parental_education, ROUND(AVG(average_score), 2) AS avg_score
    FROM students
    GROUP BY parental_education
    ORDER BY avg_score DESC
""").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT gender, lunch, ROUND(AVG(math_score), 2) AS avg_math
    FROM students
    GROUP BY gender, lunch
    ORDER BY avg_math DESC
""").show()

In [ ]:
spark.sql("""
    SELECT *
    FROM students
    ORDER BY average_score DESC
    LIMIT 10
""").show()

## 11. Data Visualization

In [ ]:
import matplotlib.pyplot as plt

prep = df.groupBy("test_prep").agg(
    avg("math_score").alias("Math"), avg("reading_score").alias("Reading"),
    avg("writing_score").alias("Writing")).toPandas().set_index("test_prep")
prep.plot(kind="bar", figsize=(7, 4))
plt.ylabel("Average score")
plt.title("Test Preparation vs Scores")
plt.xticks(rotation=0)
plt.show()

## 12. Saving Results Back to HDFS

In [ ]:
df.write.mode("overwrite").parquet("hdfs://localhost:9000/user/student/output/students_processed")

df.groupBy("performance").count().write.mode("overwrite").csv(
    "hdfs://localhost:9000/user/student/output/performance_summary", header=True)

df.groupBy("test_prep").agg(avg("average_score").alias("avg_score")).write.mode("overwrite").csv(
    "hdfs://localhost:9000/user/student/output/testprep_summary", header=True)

!hdfs dfs -ls -R /user/student/output

In [ ]:
# Read the saved result back from HDFS to confirm the round trip
spark.read.parquet("hdfs://localhost:9000/user/student/output/students_processed").show(5)
!hdfs dfs -du -h /user/student

## 13. Project Website (interactive dashboard)
The last cell reads the processed data **from HDFS**, builds the website, and prints a public link you can open in any browser or share.

In [ ]:
# Build the website from the processed data stored in HDFS
import json, os

res = spark.read.parquet("hdfs://localhost:9000/user/student/output/students_processed").toPandas()
res["passed"] = res["passed"].astype(bool)
data_json = res.to_json(orient="records")

HTML_TEMPLATE = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Student Performance Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<style>
:root{--bg:#f6f7fb;--card:#fff;--text:#1c2030;--muted:#6b7285;--line:#e3e6ef;--accent:#4f46e5;color-scheme:light}
@media (prefers-color-scheme:dark){:root{--bg:#12141c;--card:#1b1e29;--text:#e8eaf2;--muted:#9aa1b5;--line:#2c3040;--accent:#818cf8;color-scheme:dark}}
*{box-sizing:border-box}
body{margin:0;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;background:var(--bg);color:var(--text)}
header{padding:24px 20px 8px;max-width:1200px;margin:auto}
h1{margin:0 0 4px;font-size:24px}
.sub{color:var(--muted);font-size:14px}
main{max-width:1200px;margin:auto;padding:12px 20px 40px}
.card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:16px}
.filters{display:flex;flex-wrap:wrap;gap:12px;align-items:end;margin-bottom:16px}
label{display:flex;flex-direction:column;font-size:12px;color:var(--muted);gap:4px}
select,input{background:var(--card);color:var(--text);border:1px solid var(--line);border-radius:8px;padding:8px 10px;font-size:14px}
button{background:var(--accent);color:#fff;border:0;border-radius:8px;padding:9px 14px;font-size:14px;cursor:pointer}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin-bottom:16px}
.kpi .v{font-size:26px;font-weight:600}
.kpi .l{font-size:12px;color:var(--muted)}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px;margin-bottom:16px}
.card h2{font-size:15px;margin:0 0 10px}
.row{display:flex;justify-content:space-between;align-items:center;gap:8px;flex-wrap:wrap;margin-bottom:8px}
canvas{max-height:300px}
.tw{overflow-x:auto}
table{width:100%;border-collapse:collapse;font-size:13px}
th,td{text-align:left;padding:8px;border-bottom:1px solid var(--line);white-space:nowrap}
th{color:var(--muted);font-weight:500}
.pill{padding:2px 8px;border-radius:99px;font-size:12px;background:var(--line)}
footer{color:var(--muted);font-size:12px;text-align:center;padding:16px}
</style>
</head>
<body>
<header>
  <h1>Student Performance Dashboard</h1>
  <div class="sub">Results processed with HDFS and Apache Spark · <span id="count"></span></div>
</header>
<main>
  <div class="card filters">
    <label>Gender<select id="f_gender"></select></label>
    <label>Race/ethnicity<select id="f_race_ethnicity"></select></label>
    <label>Parental education<select id="f_parental_education"></select></label>
    <label>Lunch<select id="f_lunch"></select></label>
    <label>Test prep<select id="f_test_prep"></select></label>
    <button id="reset">Reset filters</button>
  </div>

  <div class="kpis" id="kpis"></div>

  <div class="grid">
    <div class="card">
      <div class="row"><h2>Average scores by group</h2>
        <label>Group by<select id="groupBy">
          <option value="gender">Gender</option>
          <option value="race_ethnicity">Race/ethnicity</option>
          <option value="parental_education">Parental education</option>
          <option value="lunch">Lunch</option>
          <option value="test_prep">Test prep course</option>
        </select></label></div>
      <canvas id="barChart"></canvas>
    </div>
    <div class="card">
      <div class="row"><h2>Score distribution</h2>
        <label>Subject<select id="subject">
          <option value="math_score">Math</option>
          <option value="reading_score">Reading</option>
          <option value="writing_score">Writing</option>
          <option value="average_score">Overall average</option>
        </select></label></div>
      <canvas id="histChart"></canvas>
    </div>
    <div class="card">
      <h2>Performance categories</h2>
      <canvas id="pieChart"></canvas>
    </div>
    <div class="card">
      <h2>Math vs reading score</h2>
      <canvas id="scatterChart"></canvas>
    </div>
  </div>

  <div class="card">
    <div class="row"><h2>Top students (current filters)</h2>
      <label>Sort by<select id="sortBy">
        <option value="average_score">Overall average</option>
        <option value="math_score">Math</option>
        <option value="reading_score">Reading</option>
        <option value="writing_score">Writing</option>
      </select></label></div>
    <div class="tw"><table id="tbl"></table></div>
  </div>
</main>
<footer>Pipeline: dataset → HDFS → Spark (DataFrame + SQL) → results → this dashboard</footer>

<script>
const D = __DATA__;
const FIELDS = ["gender","race_ethnicity","parental_education","lunch","test_prep"];
const COLORS = ["#4f46e5","#10b981","#f59e0b","#ef4444","#06b6d4","#a855f7"];
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const avg = (a,k) => a.length ? a.reduce((s,x)=>s+x[k],0)/a.length : 0;
const $ = id => document.getElementById(id);
let charts = {};

FIELDS.forEach(f => {
  const s = $("f_"+f);
  const vals = [...new Set(D.map(d=>d[f]))].sort();
  s.innerHTML = '<option value="">All</option>' + vals.map(v=>'<option>'+v+'</option>').join("");
  s.onchange = render;
});
["groupBy","subject","sortBy"].forEach(id => $(id).onchange = render);
$("reset").onclick = () => { FIELDS.forEach(f => $("f_"+f).value = ""); render(); };

function filtered(){
  return D.filter(d => FIELDS.every(f => !$("f_"+f).value || d[f] === $("f_"+f).value));
}
function draw(id, cfg){
  if(charts[id]) charts[id].destroy();
  Chart.defaults.color = css("--muted");
  Chart.defaults.borderColor = css("--line");
  charts[id] = new Chart($(id), cfg);
}
function render(){
  const R = filtered();
  $("count").textContent = R.length + " of " + D.length + " students";
  const pass = R.length ? R.filter(d=>d.passed).length / R.length * 100 : 0;
  const kp = [["Students", R.length],["Avg math", avg(R,"math_score").toFixed(1)],
    ["Avg reading", avg(R,"reading_score").toFixed(1)],["Avg writing", avg(R,"writing_score").toFixed(1)],
    ["Pass rate", pass.toFixed(1)+"%"]];
  $("kpis").innerHTML = kp.map(k=>'<div class="card kpi"><div class="v">'+k[1]+'</div><div class="l">'+k[0]+'</div></div>').join("");

  // grouped bar
  const g = $("groupBy").value;
  const groups = [...new Set(R.map(d=>d[g]))].sort();
  const subs = [["math_score","Math"],["reading_score","Reading"],["writing_score","Writing"]];
  draw("barChart",{type:"bar",data:{labels:groups,datasets:subs.map((s,i)=>({
    label:s[1],backgroundColor:COLORS[i],data:groups.map(v=>+avg(R.filter(d=>d[g]===v),s[0]).toFixed(1))}))},
    options:{scales:{y:{beginAtZero:true,max:100}}}});

  // histogram
  const sub = $("subject").value;
  const bins = Array(10).fill(0);
  R.forEach(d => bins[Math.min(9, Math.floor(d[sub]/10))]++);
  draw("histChart",{type:"bar",data:{labels:bins.map((_,i)=>(i*10)+"–"+(i===9?100:i*10+9)),
    datasets:[{label:"Students",backgroundColor:COLORS[0],data:bins}]},
    options:{plugins:{legend:{display:false}}}});

  // performance pie
  const cats = ["Excellent","Good","Average","Poor"];
  draw("pieChart",{type:"doughnut",data:{labels:cats,datasets:[{backgroundColor:[COLORS[1],COLORS[0],COLORS[2],COLORS[3]],
    borderWidth:0,data:cats.map(c=>R.filter(d=>d.performance===c).length)}]}});

  // scatter
  draw("scatterChart",{type:"scatter",data:{datasets:[{label:"Student",backgroundColor:COLORS[0]+"99",pointRadius:3,
    data:R.map(d=>({x:d.math_score,y:d.reading_score}))}]},
    options:{plugins:{legend:{display:false}},scales:{x:{title:{display:true,text:"Math"},min:0,max:100},y:{title:{display:true,text:"Reading"},min:0,max:100}}}});

  // table
  const k = $("sortBy").value;
  const top = [...R].sort((a,b)=>b[k]-a[k]).slice(0,10);
  $("tbl").innerHTML = "<tr><th>#</th><th>Gender</th><th>Group</th><th>Parent education</th><th>Lunch</th><th>Test prep</th><th>Math</th><th>Reading</th><th>Writing</th><th>Avg</th><th>Level</th></tr>" +
    top.map((d,i)=>"<tr><td>"+(i+1)+"</td><td>"+d.gender+"</td><td>"+d.race_ethnicity+"</td><td>"+d.parental_education+"</td><td>"+d.lunch+"</td><td>"+d.test_prep+"</td><td>"+d.math_score+"</td><td>"+d.reading_score+"</td><td>"+d.writing_score+"</td><td>"+d.average_score.toFixed(1)+'</td><td><span class="pill">'+d.performance+"</span></td></tr>").join("");
}
render();
</script>
</body>
</html>
'''

os.makedirs("/content/website", exist_ok=True)
with open("/content/website/index.html", "w") as f:
    f.write(HTML_TEMPLATE.replace("__DATA__", data_json))
print("Website built with", len(res), "students")

In [ ]:
# Start the web server and open a public link
import subprocess, time, re, urllib.request, sys

# 1. local web server
subprocess.run("pkill -f 'http.server 8000'", shell=True)
server = subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--directory", "/content/website"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

# 2. public tunnel (Cloudflare quick tunnel, no signup needed)
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)

subprocess.run("pkill -f cloudflared", shell=True)
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
        break

if url:
    from IPython.display import HTML, display
    print("Your interactive website is live:")
    display(HTML(f'<a href="{url}" target="_blank" style="font-size:20px;font-weight:bold">{url}</a>'))
    print("Keep this cell running. The link stops working when the Colab session ends.")
else:
    print("Public tunnel failed, using Colab's private link instead (works for you only):")
    from google.colab import output
    output.serve_kernel_port_as_window(8000)